# Disaster Damage Assessment
**NB 09** | Hatay, Turkey (Feb 2023 earthquake) | ChangeFormer xBD 4-class

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.change_detection.changeformer import ChangeDetection


In [ ]:
BBOX        = (36.15, 36.15, 36.35, 36.35)  # Iskenderun / Hatay province
DATE_BEFORE = ('2022-10-01', '2023-01-31')   # Pre-earthquake
DATE_AFTER  = ('2023-02-15', '2023-04-30')   # Post-earthquake
PROVIDERS   = ['planetary_computer']
BANDS       = ['B02','B03','B04','B08']
CLASS_NAMES = ['No damage','Minor damage','Major damage','Destroyed']
DATA_DIR    = pathlib.Path('./results/nb09')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
pre_r  = client.search(bbox=BBOX, date_range=DATE_BEFORE, providers=PROVIDERS, cloud_cover_max=20)
post_r = client.search(bbox=BBOX, date_range=DATE_AFTER,  providers=PROVIDERS, cloud_cover_max=20)
print(f"Pre: {len(pre_r)} scenes | Post: {len(post_r)} scenes")
PRE_DIR = DATA_DIR/'pre'; PRE_DIR.mkdir(exist_ok=True)
POST_DIR = DATA_DIR/'post'; POST_DIR.mkdir(exist_ok=True)
pre_dl  = client.download(pre_r[:1],  output_dir=str(PRE_DIR),  bands=BANDS, post_process=['reproject:EPSG:32637','cog'])
post_dl = client.download(post_r[:1], output_dir=str(POST_DIR), bands=BANDS, post_process=['reproject:EPSG:32637','cog'])
pre_scene  = pre_dl[0].path  if pre_dl  and pre_dl[0].success  else None
post_scene = post_dl[0].path if post_dl and post_dl[0].success else None


In [ ]:
# Validate and preprocess both with identical parameters
PRE_READY  = DATA_DIR / 'pre_ready.tif'
POST_READY = DATA_DIR / 'post_ready.tif'
for scene, out, label in [(pre_scene, PRE_READY,'PRE'), (post_scene, POST_READY,'POST')]:
    if scene:
        rep = client.validator.validate(str(scene), required_bands=len(BANDS))
        r   = client.prepare_for_ai(str(scene), stack_bands=BANDS, bbox=BBOX,
            normalise='scale_factor', scale_factor=10000.0,
            model_type='change_detection', output_path=str(out))
        print(f"{label}: {r['steps']}  shape={r['shape']}")


In [ ]:
DAMAGE_MAP = DATA_DIR / 'damage_map.tif'
if PRE_READY.exists() and POST_READY.exists():
    cd = ChangeDetection(model_variant='changeformer', in_channels=len(BANDS), num_classes=4)
    cd.build()
    res = cd.detect(str(PRE_READY), str(POST_READY), output_path=str(DAMAGE_MAP))
    print(f"Damage coverage: {res['change_pct']:.2f}%")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED = DATA_DIR/'damage_sieved.tif'; COG_OUT = DATA_DIR/'damage_cog.tif'
VECTOR = DATA_DIR/'damage_polygons.geojson'
if DAMAGE_MAP.exists():
    client.postprocess.sieve_filter(str(DAMAGE_MAP), min_pixels=10, output_path=str(SIEVED))
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR), min_area_m2=50.0)
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    print(f"{'Cls':<4} {'Damage level':<16} {'Area(ha)':>10} {'Cover':>8}")
    print('-'*42)
    for i, name in enumerate(CLASS_NAMES):
        inf = stats.get(i, {})
        print(f"{i:<4} {name:<16} {inf.get('area_ha',0):>10.1f} {inf.get('pct',0):>7.1f}%")


**NB09 — Disaster Assessment** complete.
- Study area: Hatay Turkey earthquake
- Sensor: Sentinel-2 bi-temporal
- Model: ChangeFormer 4-class xBD
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG